In [3]:
import pandas as pd

# Load CSVs
plans = pd.read_csv("../data/plans.csv")
claims = pd.read_csv("../data/claims.csv")

# Inspect
print(plans.head())
print(plans.info())

print(claims.head())
print(claims.info())

# Remove duplicate rows
plans = plans.drop_duplicates()
claims = claims.drop_duplicates()

# Remove rows with missing values
plans = plans.dropna()
claims = claims.dropna()

# Convert date column to datetime
claims["date_filed"] = pd.to_datetime(claims["date_filed"])

  plan_id   plan_name  monthly_premium  annual_deductible  copay_pct  \
0    P101    Gold PPO              500               2000         10   
1    P102  Silver HMO              300               1500         20   
2    P103  Bronze HMO              150               1000         30   

  coverage_type network_tier  
0           PPO         Gold  
1           HMO       Silver  
2           HMO       Bronze  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   plan_id            3 non-null      object
 1   plan_name          3 non-null      object
 2   monthly_premium    3 non-null      int64 
 3   annual_deductible  3 non-null      int64 
 4   copay_pct          3 non-null      int64 
 5   coverage_type      3 non-null      object
 6   network_tier       3 non-null      object
dtypes: int64(3), object(4)
memory usage: 300.0+ bytes
None
  cl

In [2]:
import os

print(os.getcwd())

/Users/anushachennu/projects/coverage-chatbot-api/ABCohort/notebook


In [4]:
import pandas as pd
import sqlite3

# Load CSVs
plans = pd.read_csv("../data/plans.csv")
claims = pd.read_csv("../data/claims.csv")

# Clean data
plans = plans.drop_duplicates()
claims = claims.drop_duplicates()

plans = plans.dropna()
claims = claims.dropna()

claims["date_filed"] = pd.to_datetime(claims["date_filed"])

# Create database
conn = sqlite3.connect("../coverage.db")

# Save tables
plans.to_sql("plans", conn, if_exists="replace", index=False)
claims.to_sql("claims", conn, if_exists="replace", index=False)

# Close database
conn.close()

print("Database created successfully!")

Database created successfully!


In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../coverage.db")

In [7]:
# What's the deductible on the Gold PPO plan?

query = """
SELECT *
FROM plans;
"""

pd.read_sql(query, conn)



,plan_id,plan_name,monthly_premium,annual_deductible,copay_pct,coverage_type,network_tier
0,P101,Gold PPO,500,2000,10,PPO,Gold
1,P102,Silver HMO,300,1500,20,HMO,Silver
2,P103,Bronze HMO,150,1000,30,HMO,Bronze


In [10]:
#How many claims are pending for member M1001?

query = """
SELECT COUNT(*) AS pending_claims
FROM claims
WHERE member_id='M1001'
AND status='Pending';
"""

pd.read_sql(query, conn)

,pending_claims
0,1


In [9]:

query = """
SELECT annual_deductible
FROM plans
WHERE plan_name = 'Gold PPO';
"""

pd.read_sql(query, conn)

,annual_deductible
0,2000


In [13]:
#Which plans have a monthly premium under $400?
query = """
SELECT plan_name,monthly_premium 
FROM plans 
WHERE monthly_premium<400
"""
pd.read_sql(query, conn)

,plan_name,monthly_premium
0,Silver HMO,300
1,Bronze HMO,150


In [16]:
#- A JOIN between claims and plans - 
query = """
SELECT
    c.claim_id,
    p.plan_name,
    c.claim_amount
FROM claims AS c
JOIN plans AS p
ON c.plan_id = p.plan_id;
"""

pd.read_sql(query, conn)


,claim_id,plan_name,claim_amount
0,C1001,Gold PPO,250
1,C1002,Gold PPO,1200
2,C1003,Silver HMO,150
3,C1004,Silver HMO,900
4,C1005,Bronze HMO,50


In [18]:
#A top-N query - Which procedures are claimed the most?

query= """
SELECT
    procedure,
    COUNT(*) AS total_claims
FROM claims
GROUP BY procedure
ORDER BY total_claims DESC
LIMIT 5;
"""

pd.read_sql(query, conn)



,procedure,total_claims
0,X-ray,3
1,Surgery,2
